# Loading DGCI embeddings:

In [ ]:
%matplotlib widget    
import numpy as np                                          # already imported in cell 1
from sklearn.cluster import KMeans                          # the clustering used in the paper
from sklearn.decomposition import PCA                       # only for the 2D plot
from sklearn.metrics import silhouette_score                # cluster quality without labels
from sklearn.metrics import adjusted_rand_score             # agreement between two clusterings
import matplotlib.pyplot as plt   
import pandas as pd                                              # tables keyed by subject
import morphology_projection as mp  

codes_path = "/home/ids/gmargari-24/airway_project/DGCI/logs/unregistered_128_2M_4k/latent_codes.npy"
enc_output = np.load(codes_path).astype(np.float64)         # (419, 128); float64 keeps ward's sums stable

In [ ]:
codes      = enc_output                                     # (419, 128)
dim_median = np.median(codes, axis=0)                       # typical value of each dimension
dim_mad    = np.median(np.abs(codes - dim_median), axis=0)  # robust spread, ignores outlier shapes
mad_order  = np.argsort(dim_mad)[::-1]                      # most to least spread ([::-1] = reverse)

plt.figure(figsize=(8, 4))                                  # one wide panel
plt.semilogy(dim_mad[mad_order], marker=".")                # log scale exposes the gap
plt.xlabel("dimension rank")                                # sorted, not original index
plt.ylabel("MAD across shapes (log)")                       # spread per dimension
plt.show()                                                  # render

In [ ]:
pca_full      = PCA().fit(codes)                              # all 128 principal directions
eigvals       = pca_full.explained_variance_                  # variance along each direction
participation = eigvals.sum() ** 2 / (eigvals ** 2).sum()     # effective number of dimensions
cum_share     = np.cumsum(pca_full.explained_variance_ratio_) # cumulative variance explained
n_95          = np.searchsorted(cum_share, 0.95) + 1          # directions needed for 95% variance
print(f"participation ratio {participation:.1f}, 95% variance in {n_95} directions")  # summary

plt.figure(figsize=(8, 4))                                  # one wide panel
plt.semilogy(eigvals, marker=".")                           # look for where the spectrum drops
plt.xlabel("principal component")                           # ordered by variance
plt.ylabel("variance (log)")                                # per-direction variance
plt.show()                                                  # render

In [ ]:
active_dims     = np.where(dim_mad > 3e-4)[0]                        # threshold sits in the gap
borderline_dims = np.where((dim_mad > 1e-4) & (dim_mad <= 3e-4))[0]  # ranks ~46–49
print(len(active_dims), active_dims)                                 # expect roughly 45
print(len(borderline_dims), borderline_dims)                         # confirm these with ablation

In [ ]:
from sklearn.cluster import KMeans                          # clustering
from sklearn.decomposition import PCA                       # 2D plot only
from sklearn.metrics import silhouette_score                # cluster quality without labels
from sklearn.metrics import adjusted_rand_score             # agreement between two clusterings
import matplotlib.pyplot as plt                             # plotting

dim_median   = np.median(enc_output, axis=0)                 # typical value per dimension
dim_mad      = np.median(np.abs(enc_output - dim_median), axis=0)  # robust spread per dimension
active_dims  = np.where(dim_mad > 3e-4)[0]                   # threshold sits in the gap of your plot
wide_dims    = np.where(dim_mad > 1e-4)[0]                   # also includes the borderline ones

codes_active = enc_output[:, active_dims]                   # (419, ~45)
codes_active = enc_output                                   # (419, ~45)
print(codes_active.shape, len(wide_dims))                   # confirm the counts

In [ ]:
k_values    = list(range(2, 11))                            # candidate numbers of clusters
inertias    = []                                            # within-cluster spread per K
silhouettes = []                                            # separation score per K

for k in k_values:                                          # one fit per candidate K
    km  = KMeans(n_clusters=k, n_init=20, random_state=0)   # n_init = restarts, best one kept
    ids = km.fit_predict(codes_active)                      # cluster the active dimensions only
    inertias.append(km.inertia_)                            # for the elbow plot
    silhouettes.append(silhouette_score(codes_active, ids)) # higher = better separated

fig, axes = plt.subplots(1, 2, figsize=(10, 4))             # elbow and silhouette side by side
axes[0].plot(k_values, inertias, marker="o")                # look for the bend
axes[0].set_xlabel("K")                                     # number of clusters
axes[0].set_ylabel("Inertia")                               # lower = tighter
axes[1].plot(k_values, silhouettes, marker="o")             # look for the peak
axes[1].set_xlabel("K")                                     # number of clusters
axes[1].set_ylabel("Silhouette")                            # higher = better separated
plt.show()                                                  # render

In [ ]:
n_clusters = 4                                        # replace with the K chosen above

kmeans      = KMeans(n_clusters=n_clusters, n_init=20, random_state=0) # 
cluster_ids = kmeans.fit_predict(codes_active)              # cluster id for each of the 419 shapes
print(np.bincount(cluster_ids))                             # shapes per cluster (bincount = counts)

pca        = PCA(n_components=2)                            # plot axes from the active dims
codes_2d   = pca.fit_transform(codes_active)                # (419, 2)
centers_2d = pca.transform(kmeans.cluster_centers_)         # centroids on the same axes
print(pca.explained_variance_ratio_)                        # how much the 2D view shows

plt.figure(figsize=(6, 5))                                  # single panel like Fig. 3
plt.scatter(codes_2d[:, 0], codes_2d[:, 1], c=cluster_ids, cmap="tab10", alpha=0.6)  # color = cluster
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="o", s=100)  # centroids
plt.xlabel("PCA1")                                          # first component
plt.ylabel("PCA2")                                          # second component
plt.show()                                                  # render

In [ ]:
def plot_clusters(xy, title, centers=None):
    """Scatter 2D points colored by K-means cluster, with a legend."""
    plt.figure(figsize=(7, 6))                              # slightly larger canvas
    for c in range(n_clusters):                             # one call per cluster gives a legend
        pts = xy[cluster_ids == c]                          # points in cluster c
        # smaller markers without edges overlap less
        plt.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.7, edgecolors="none", label=f"cluster {c}")
    if centers is not None:                                 # t-SNE has no centroids to draw
        plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="x", s=120)  # centroids
    plt.title(title)                                        # which projection this is
    plt.legend()                                            # cluster names

In [ ]:
# low = np.percentile(codes_2d, 1, axis=0)                    # 1st percentile on each axis
# high = np.percentile(codes_2d, 99, axis=0)                  # 99th percentile on each axis
# margin = 0.1 * (high - low)                                 # a little padding around the box

# plot_clusters(codes_2d, "PCA (zoomed)", centers_2d)         # same data as before
# plt.xlim(low[0] - margin[0], high[0] + margin[0])           # crop the far outliers horizontally
# plt.ylim(low[1] - margin[1], high[1] + margin[1])           # and vertically
# plt.xlabel("PCA1")                                          # first component
# plt.ylabel("PCA2")                                          # second component
# plt.show()                                                  # render

### Active latent codes
       │
       ├──────────────► K-means ───► cluster labels
       │
       └──────────────► Kernel PCA ─► 3D visualization

In [ ]:
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis  # supervised projection

# # shrinkage="auto" keeps the covariance estimate stable with ~45 dims and 419 shapes
# lda         = LinearDiscriminantAnalysis(n_components=2, solver="eigen", shrinkage="auto")
# codes_lda   = lda.fit_transform(codes_active, cluster_ids)  # at most K-1 = 3 components available
# centers_lda = lda.transform(kmeans.cluster_centers_)        # centroids on the same axes

# plot_clusters(codes_lda, "LDA on K-means clusters", centers_lda)  # separation-maximizing view
# plt.xlabel("LD1")                                           # best separating direction
# plt.ylabel("LD2")                                           # second best
# plt.show()                                                  # render

In [ ]:
import pandas as pd                                              # tables keyed by subject
import morphology_projection as mp                               # helper module, same folder as the notebook

log_dir         = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k"  # DGCI run folder
latent_csv      = f"{log_dir}/latent_codes.csv"                  # subject ids in latent_codes.npy row order
metrics_csv     = f"{log_dir}/morphology/mean_morphological_features.csv"  # saved by clustering.ipynb
tree_counts_csv = "/home/ids/gmargari-24/airway_project/airmorph_cases/tree_counts.csv"  # branch/generation counts
kmeans_dir      = f"{log_dir}/morphology_kmeans_k{n_clusters}"   # own folder, so ward outputs aren't overwritten

subjects = pd.read_csv(latent_csv)["subject"].tolist()           # row i here == row i of enc_output
assert len(subjects) == len(cluster_ids), "labels and subjects differ in length"  # catch a wrong file early

metrics = pd.read_csv(metrics_csv)                               # subject + the 6 morphology means
assert metrics["subject"].tolist() == subjects, "metrics not in latent row order"  # alignment guard

counts = mp.build_tree_count_table(subjects, tree_counts_csv)    # subject + n_branches + max_generation

# 8 means per K-means cluster, plus a Kruskal-Wallis test per metric
wide, detailed, merged = mp.cluster_summary(cluster_ids, metrics, counts, save_dir=kmeans_dir)

print("\nmean of every metric per K-means cluster:")                # header for the compact table
print(wide.to_string())                                          # n + the 8 means, one row per cluster

print("\nwith spread and valid counts:")                         # header for the detailed table
print(detailed.to_string())                                      # mean / sd / valid n per metric

heatmap_path = mp.plot_cluster_heatmap(merged, save_dir=kmeans_dir)  # same means on a common SD scale
print(heatmap_path)                                              # where the PNG was written

                  DGCI latent codes
                         │
                         ▼
               Active latent dimensions
                         │
               ┌─────────┴─────────┐
               │                   │
               ▼                   ▼
            K-means             Other clustering
               │
               ▼
          Cluster labels
               │
       ┌───────┼────────┐
       │       │        │
       ▼       ▼        ▼
      PCA    KPCA     UMAP
      3D      3D       3D

In [ ]:
def centroid_projection(data, centers):
    """Coordinates of data and centroids in the 3D space through the centroids."""
    center_mean = centers.mean(axis=0)                      # middle point of the 4 centroids
    offsets = centers - center_mean                         # each centroid relative to that middle
    _, _, vt = np.linalg.svd(offsets, full_matrices=False)  # directions the offsets span (rows of vt)
    basis = vt[:3]                                          # 4 centroids span at most 3 directions
    data_3d = (data - center_mean) @ basis.T                # points in 3D (@ = matrix product)
    centers_3d = offsets @ basis.T                          # centroids on the same axes
    return data_3d, centers_3d                              # both share one coordinate system


def plot_clusters_3d(xyz, labels, title, centers=None):
    """3D scatter colored by cluster, cropped to the 1st-99th percentile."""
    fig = plt.figure(figsize=(8, 7))                        # one large panel
    ax = fig.add_subplot(projection="3d")                   # 3D axes
    for c in np.unique(labels):                             # one call per cluster (works for any K)
        pts = xyz[labels == c]                              # points in cluster c
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=14, alpha=0.7, label=f"cluster {c}")
    if centers is not None:                                 # draw centroids when available
        ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2], c="black", marker="x", s=150)
    low = np.percentile(xyz, 1, axis=0)                     # crop far outliers, as in your 2D zoom
    high = np.percentile(xyz, 99, axis=0)                   # upper bound per axis
    ax.set_xlim(low[0], high[0])                            # axis 1 range
    ax.set_ylim(low[1], high[1])                            # axis 2 range
    ax.set_zlim(low[2], high[2])                            # axis 3 range
    ax.set_title(title)                                     # which view this is
    ax.legend()                                             # cluster names
    return ax                                               # lets the caller change the angle

In [ ]:
# codes_3d, centers_3d = centroid_projection(codes_active, kmeans.cluster_centers_)  # real data

# # sanity check: nearest centroid in 3D should equal the 46-D label for every shape
# dists_3d = np.linalg.norm(codes_3d[:, None, :] - centers_3d[None, :, :], axis=2)  # (419, 4) distances
# print("same assignment in 3D:", (dists_3d.argmin(axis=1) == cluster_ids).mean())  # expect 1.0

# ax = plot_clusters_3d(codes_3d, cluster_ids, "K-means clusters, centroid subspace", centers_3d)
# ax.view_init(elev=20, azim=40)                              # camera angle (elevation, rotation), try others
# plt.show()                                                  # render

In [ ]:
# ============================================================
# 3D PCA / 3D Kernel PCA / 3D UMAP
# Visualization of the SAME K-means cluster labels
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# 1. 3D PCA
# ------------------------------------------------------------

pca_3d = PCA(n_components=3)

codes_pca_3d = pca_3d.fit_transform(codes_active)
centers_pca_3d = pca_3d.transform(kmeans.cluster_centers_)

print("3D PCA explained variance ratio:")
print(pca_3d.explained_variance_ratio_)
print("Total:", pca_3d.explained_variance_ratio_.sum())


fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(projection="3d")

for c in range(n_clusters):
    pts = codes_pca_3d[cluster_ids == c]

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        pts[:, 2],
        s=18,
        alpha=0.7,
        label=f"cluster {c}"
    )

ax.scatter(
    centers_pca_3d[:, 0],
    centers_pca_3d[:, 1],
    centers_pca_3d[:, 2],
    c="black",
    marker="x",
    s=180,
    linewidths=3
)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("K-means clusters — 3D PCA")
ax.legend()

ax.view_init(elev=20, azim=40)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 2. 3D Kernel PCA
# ------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA

# Standardize the active latent representation
scaler = StandardScaler()

codes_active_scaled = scaler.fit_transform(codes_active)


# RBF Kernel PCA
kpca_3d = KernelPCA(
    n_components=3,
    kernel="rbf",
    gamma=0.03,
    random_state=42
)

codes_kpca_3d = kpca_3d.fit_transform(codes_active_scaled)


fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(projection="3d")

for c in range(n_clusters):
    pts = codes_kpca_3d[cluster_ids == c]

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        pts[:, 2],
        s=18,
        alpha=0.7,
        label=f"cluster {c}"
    )

ax.set_xlabel("Kernel PC1")
ax.set_ylabel("Kernel PC2")
ax.set_zlabel("Kernel PC3")
ax.set_title("K-means clusters — 3D Kernel PCA")
ax.legend()

ax.view_init(elev=20, azim=40)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 3. 3D UMAP
# ------------------------------------------------------------

import umap.umap_ as umap

umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=30,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

codes_umap_3d = umap_3d.fit_transform(codes_active)


fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(projection="3d")

for c in range(n_clusters):
    pts = codes_umap_3d[cluster_ids == c]

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        pts[:, 2],
        s=18,
        alpha=0.7,
        label=f"cluster {c}"
    )

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_zlabel("UMAP3")
ax.set_title("K-means clusters — 3D UMAP")
ax.legend()

ax.view_init(elev=20, azim=40)

plt.tight_layout()
plt.show()

## Kernel K-means (RBF kernel)

K-means groups shapes by straight-line distance, so every border between two clusters is a flat hyperplane.
Kernel K-means runs the same algorithm on RBF similarities k(x, y) = exp(-γ‖x − y‖²), which lets borders curve.
It is a **clustering method**, so K has to be chosen again.

Choices made here:
- **Same space as K-means** (`codes_active`, not standardized), so the two label sets are directly comparable.
- **γ from the median rule**: a typical pair of shapes gets similarity ≈ 0.37.
- **Sanity check**: with a very wide kernel (small γ), kernel K-means should reproduce plain K-means.
- scikit-learn has no kernel K-means, so it is implemented below (≈ 40 lines, no new dependency).

In [ ]:
from sklearn.metrics import pairwise_distances              # all pairwise shape distances
from sklearn.metrics.pairwise import rbf_kernel             # similarity matrix exp(-gamma * d^2)


def median_rule_gamma(data):
    """Gamma such that a typical pair of shapes has similarity ~0.37."""
    sq_dists = pairwise_distances(data) ** 2                # (n, n) squared distances
    typical = np.median(sq_dists[sq_dists > 0])             # typical pair, skipping self-distances
    return 1.0 / typical                                    # exp(-1) ≈ 0.37 for a typical pair


def kernel_distance_matrix(K):
    """Pairwise distances in the kernel feature space, for silhouette scores."""
    diag = np.diag(K)                                       # each shape's similarity with itself
    d2 = diag[:, None] + diag[None, :] - 2 * K              # ||phi(x) - phi(y)||^2 via the kernel trick
    d2 = np.clip(d2, 0, None)                               # remove tiny negatives from rounding
    dist = np.sqrt(d2)                                      # squared distance -> distance
    np.fill_diagonal(dist, 0.0)                             # exact zeros on the diagonal
    return dist                                             # (n, n), usable with metric="precomputed"


def cluster_distances(K, labels, n_clusters):
    """Squared feature-space distance from every shape to every cluster mean, shape (n, k)."""
    n = K.shape[0]                                          # number of shapes
    diag = np.diag(K)                                       # self-similarities
    dists = np.full((n, n_clusters), np.inf)                # inf = empty cluster, never chosen
    for c in range(n_clusters):                             # one column per cluster
        members = labels == c                               # boolean mask of cluster c
        size = members.sum()                                # shapes in cluster c
        if size == 0:
            continue                                        # empty cluster stays at inf
        cross = K[:, members].sum(axis=1) / size            # mean similarity of each shape to cluster c
        within = K[np.ix_(members, members)].sum() / size**2  # mean similarity inside c (ix_ = sub-block)
        dists[:, c] = diag - 2 * cross + within             # ||phi(x) - mean_c||^2, no explicit phi
    return dists


def kernel_kmeans_once(K, n_clusters, rng, max_iter=100):
    """One run: k-means++ style seeds, then assign/update until labels stop changing."""
    n = K.shape[0]                                          # number of shapes
    diag = np.diag(K)                                       # self-similarities
    seeds = [int(rng.integers(n))]                          # first seed: a random shape
    for _ in range(n_clusters - 1):                         # pick the remaining seeds
        seed_d2 = diag[:, None] + diag[seeds][None, :] - 2 * K[:, seeds]  # (n, seeds) squared distances
        nearest = np.clip(seed_d2.min(axis=1), 0, None)     # distance to the closest seed so far
        probs = nearest / nearest.sum()                     # far shapes are more likely picked
        seeds.append(int(rng.choice(n, p=probs)))           # k-means++ pick
    seed_d2 = diag[:, None] + diag[seeds][None, :] - 2 * K[:, seeds]  # distances to all seeds
    labels = seed_d2.argmin(axis=1)                         # start: each shape joins its nearest seed

    for _ in range(max_iter):                               # usually converges in < 30 steps
        dists = cluster_distances(K, labels, n_clusters)    # distances to current cluster means
        new_labels = dists.argmin(axis=1)                   # reassign to the nearest mean
        own = dists[np.arange(n), new_labels]               # each shape's distance to its own cluster
        for c in range(n_clusters):                         # never let a cluster disappear
            if np.any(new_labels == c):
                continue                                    # cluster c still has members
            worst = own.argmax()                            # shape that fits its cluster worst
            new_labels[worst] = c                           # it restarts the empty cluster
            own[worst] = -np.inf                            # don't reuse it for another empty cluster
        if np.array_equal(new_labels, labels):
            break                                           # converged
        labels = new_labels                                 # next iteration

    dists = cluster_distances(K, labels, n_clusters)        # final distances
    objective = dists[np.arange(n), labels].sum()           # within-cluster spread (kernel "inertia")
    return labels, objective


def kernel_kmeans(K, n_clusters, n_init=20, random_state=0):
    """Best of n_init runs (lowest objective), like KMeans(n_init=...)."""
    rng = np.random.default_rng(random_state)               # reproducible restarts
    best_labels = None                                      # filled by the first run
    best_objective = np.inf                                 # any run beats this
    for _ in range(n_init):                                 # independent restarts
        labels, objective = kernel_kmeans_once(K, n_clusters, rng)
        if objective < best_objective:                      # keep the tightest solution
            best_labels = labels                            # labels of the best run
            best_objective = objective                      # its objective
    return best_labels, best_objective

In [ ]:
kk_gamma = median_rule_gamma(codes_active)                  # kernel width for this data scale
K_rbf    = rbf_kernel(codes_active, gamma=kk_gamma)         # (419, 419) similarities
D_rbf    = kernel_distance_matrix(K_rbf)                    # feature-space distances for silhouettes
print(f"median-rule gamma: {kk_gamma:.3g}")                 # expect around 1e4 for codes at ~1e-3 scale

# sanity check: a very wide kernel turns kernel K-means into plain K-means.
# Compare the Euclidean spread (inertia) of its labels with KMeans' own; ARI can be low on
# weakly clustered data, because even plain K-means lands on different partitions per start.
def euclidean_inertia(data, labels):
    """Sum of squared distances of shapes to their own cluster mean (KMeans.inertia_)."""
    total = 0.0                                             # running sum over clusters
    for c in np.unique(labels):                             # each cluster separately
        members = data[labels == c]                         # shapes in cluster c
        total += ((members - members.mean(axis=0)) ** 2).sum()  # spread around the cluster mean
    return total


K_wide      = rbf_kernel(codes_active, gamma=0.01 * kk_gamma)  # 100x wider kernel
wide_ids, _ = kernel_kmeans(K_wide, n_clusters)             # same K as your K-means
ratio       = euclidean_inertia(codes_active, wide_ids) / kmeans.inertia_  # 1.00 = equally tight
print(f"wide kernel inertia / K-means inertia: {ratio:.3f}  (expect ~1.00)")

In [ ]:
k_values_kk   = list(range(2, 11))                          # same candidates as the K-means sweep
kk_objectives = []                                          # kernel within-cluster spread per K
kk_sil_kernel = []                                          # silhouette in the kernel feature space
kk_sil_euclid = []                                          # silhouette in the original active space

for k in k_values_kk:                                       # one fit per candidate K
    ids, objective = kernel_kmeans(K_rbf, k)                # best of 20 restarts
    kk_objectives.append(objective)                         # for the elbow plot
    kk_sil_kernel.append(silhouette_score(D_rbf, ids, metric="precomputed"))  # space it optimizes
    kk_sil_euclid.append(silhouette_score(codes_active, ids))  # comparable with K-means numbers

fig, axes = plt.subplots(1, 2, figsize=(11, 4))             # elbow and silhouette side by side
axes[0].plot(k_values_kk, kk_objectives, marker="o")        # look for the bend
axes[0].set_xlabel("K")                                     # number of clusters
axes[0].set_ylabel("Kernel objective")                      # lower = tighter
axes[0].set_title("Kernel K-means elbow")                   # panel name
axes[1].plot(k_values_kk, kk_sil_kernel, marker="o", label="kernel K-means (kernel space)")
axes[1].plot(k_values_kk, kk_sil_euclid, marker="o", label="kernel K-means (active space)")
axes[1].plot(k_values, silhouettes, marker="o", linestyle="--", label="K-means (active space)")
axes[1].set_xlabel("K")                                     # number of clusters
axes[1].set_ylabel("Silhouette")                            # higher = better separated
axes[1].legend()                                            # which line is which
plt.tight_layout()                                          # avoid overlapping labels
plt.show()                                                  # render

In [ ]:
kk_clusters = 4                                             # replace with the K chosen from the sweep above

kk_ids, _ = kernel_kmeans(K_rbf, kk_clusters)               # final kernel K-means labels
print("shapes per cluster:", np.bincount(kk_ids))           # watch for tiny or giant clusters

kk_ari = adjusted_rand_score(cluster_ids, kk_ids)           # agreement with your K-means labels
print(f"ARI vs K-means: {kk_ari:.2f}  (near 1 = same grouping, near 0 = unrelated)")

# rows = K-means cluster, columns = kernel K-means cluster (label numbers are arbitrary)
kmeans_col = pd.Series(cluster_ids, name="K-means")         # named so the table has labels
kernel_col = pd.Series(kk_ids, name="kernel K-means")       # same
print(pd.crosstab(kmeans_col, kernel_col))                  # how shapes moved between methods

In [ ]:
# 1) stability: does a different random start give the same partition?
seed_aris = []                                              # ARI of each reseeded run vs kk_ids
for seed in range(1, 6):                                    # 5 extra independent fits
    reseeded_ids, _ = kernel_kmeans(K_rbf, kk_clusters, random_state=seed)
    seed_aris.append(adjusted_rand_score(kk_ids, reseeded_ids))
print(f"stability across seeds: mean ARI {np.mean(seed_aris):.2f}  (low = clusters are not robust)")

# 2) gamma sensitivity: does the grouping depend on the kernel width?
for factor in [0.1, 1, 10]:                                 # fixed grid, report all
    K_f = rbf_kernel(codes_active, gamma=factor * kk_gamma)  # kernel at this width
    ids_f, _ = kernel_kmeans(K_f, kk_clusters)              # cluster with it
    ari_kmeans = adjusted_rand_score(cluster_ids, ids_f)    # vs plain K-means
    ari_median = adjusted_rand_score(kk_ids, ids_f)         # vs the median-rule result
    print(f"gamma x{factor:<4}: sizes {np.bincount(ids_f, minlength=kk_clusters)}, "
          f"ARI vs K-means {ari_kmeans:.2f}, vs median-rule {ari_median:.2f}")

# 3) no-cluster baseline: a Gaussian cloud with the same mean and covariance
rng_base = np.random.default_rng(0)                         # fixed seed, reproducible baseline
fake = rng_base.multivariate_normal(codes_active.mean(axis=0),
                                    np.cov(codes_active, rowvar=False),  # rowvar=False: columns = dims
                                    size=len(codes_active))  # same number of shapes
K_fake = rbf_kernel(fake, gamma=median_rule_gamma(fake))    # same kernel recipe
fake_ids, _ = kernel_kmeans(K_fake, kk_clusters)            # same clustering recipe
fake_sil = silhouette_score(kernel_distance_matrix(K_fake), fake_ids, metric="precomputed")
real_sil = silhouette_score(D_rbf, kk_ids, metric="precomputed")  # your data, same score
print(f"kernel-space silhouette: real {real_sil:.3f}, no-cluster baseline {fake_sil:.3f}")

In [ ]:
from sklearn.decomposition import KernelPCA                 # 3D view in the same kernel space

kk_kpca = KernelPCA(n_components=3, kernel="rbf", gamma=kk_gamma)  # same kernel as the clustering
codes_kk_3d = kk_kpca.fit_transform(codes_active)           # (419, 3), unscaled active dims

ax = plot_clusters_3d(codes_kk_3d, kk_ids, f"Kernel K-means (K={kk_clusters}), kernel PCA view")
ax.view_init(elev=20, azim=40)                              # camera angle, rotate with the mouse
plt.show()                                                  # render

ax = plot_clusters_3d(codes_kk_3d, cluster_ids, "K-means labels, same kernel PCA view")
ax.view_init(elev=20, azim=40)                              # identical angle for comparison
plt.show()                                                  # render

In [ ]:
# needs the cell that loads metrics and counts (the K-means morphology cell) to have run
kernel_dir = f"{log_dir}/morphology_kernel_kmeans_k{kk_clusters}"  # own folder, K-means outputs untouched

kk_wide, kk_detailed, kk_merged = mp.cluster_summary(kk_ids, metrics, counts, save_dir=kernel_dir)

print("\nmean of every metric per kernel K-means cluster:")  # header for the compact table
print(kk_wide.to_string())                                  # n + the 8 means, one row per cluster

kk_heatmap_path = mp.plot_cluster_heatmap(kk_merged, save_dir=kernel_dir)  # same means on a common SD scale
print(kk_heatmap_path)                                      # where the PNG was written

# 2d

In [ ]:
codes_kk_2d = codes_kk_3d[:, :2]                            # top 2 kernel PCs = first 2 columns of the 3D fit

kpca_all = KernelPCA(kernel="rbf", gamma=kk_gamma).fit(codes_active)  # all components, only to measure shares
eig = kpca_all.eigenvalues_                                 # kernel variance per component, largest first
print(f"kernel variance shown: 2D {eig[:2].sum() / eig.sum():.1%}, 3D {eig[:3].sum() / eig.sum():.1%}")

low = np.percentile(codes_kk_2d, 1, axis=0)                 # 1st percentile per axis
high = np.percentile(codes_kk_2d, 99, axis=0)               # 99th percentile per axis
margin = 0.1 * (high - low)                                 # padding around the crop box

panels = [(kk_ids, f"Kernel K-means (K={kk_clusters})"),    # left: labels made in kernel space
          (cluster_ids, "K-means labels")]                  # right: labels made in the original space

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)  # same map, same axis ranges
for ax, (labels, title) in zip(axes, panels):               # one panel per labeling
    for c in np.unique(labels):                             # one call per cluster gives a legend
        pts = codes_kk_2d[labels == c]                      # points in cluster c
        ax.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.7, edgecolors="none", label=f"cluster {c}")
    ax.set_xlim(low[0] - margin[0], high[0] + margin[0])    # crop far outliers horizontally
    ax.set_ylim(low[1] - margin[1], high[1] + margin[1])    # and vertically
    ax.set_xlabel("Kernel PC1")                             # largest-spread direction in kernel space
    ax.set_title(f"{title}, 2D kernel PCA view")            # which labels color this panel
    ax.legend()                                             # cluster names
axes[0].set_ylabel("Kernel PC2")                            # shared y-axis, label once
plt.tight_layout()                                          # avoid overlapping titles
plt.show()                                                  # render

In [ ]:
# --- UMAP and t-SNE of the active codes: kernel K-means vs K-means labels ---------
import os                                                       # create the output folder if missing
import umap                                                     # umap-learn package
from sklearn.manifold import TSNE                               # t-SNE

X_vis = np.asarray(codes_active, dtype=np.float64)              # the exact matrix both methods clustered (419, 46)
assert len(kk_ids) == len(cluster_ids) == X_vis.shape[0], "labels and codes have different lengths"

# --- 1. project to 2D once (same settings as 1_clustering.ipynb) --------------------
umap_2d = umap.UMAP(n_neighbors=35, min_dist=0.0125, n_components=2, metric="euclidean",random_state=42).fit_transform(X_vis)       # neighbourhood-preserving map
tsne_2d = TSNE(n_components=2, perplexity=30, init="pca", random_state=0, max_iter=1000).fit_transform(X_vis)              # neighbourhood-preserving map

# --- 2. fixed colour + marker per cluster id, identical in every panel --------------
palette = ["#2684f6", "#ab380b", "#11db94", "#f2b227", "#fc0c68", "#008300", "#4a3aa7", "#1c0404"]
markers = ["o"] * 8              # shape backs up colour (colour-blind safe)

rows = [(kk_ids, f"kernel K-means (K={kk_clusters})"),          # top row: kernel K-means labels
        (cluster_ids, f"K-means (K={n_clusters})")]             # bottom row: K-means labels
max_k = max(len(np.unique(ids)) for ids, _ in rows)             # largest number of clusters to style
assert max_k <= len(palette), "more clusters than colours: extend palette and markers"

fig, axes = plt.subplots(2, 2, figsize=(18, 15))                # rows = labelling, columns = projection
for r, (ids, method) in enumerate(rows):                        # one row per clustering method
    ids = np.asarray(ids).astype(int)                           # plain integer labels
    order = sorted(np.unique(ids), key=lambda c: -(ids == c).sum())  # big clusters first, small ones on top
    for col, (coords, name) in enumerate([(umap_2d, "UMAP"), (tsne_2d, "t-SNE")]):
        ax = axes[r, col]                                       # current panel
        for c in order:                                         # draw cluster by cluster
            m = ids == c                                        # members of cluster c
            ax.scatter(coords[m, 0], coords[m, 1], c=palette[c], marker=markers[c],
                       s=40 if m.sum() > 20 else 90,            # enlarge small clusters so they stay visible
                       alpha=0.85, edgecolors="white", linewidths=0.6,
                       label=f"cluster {c} (n={m.sum()})")
        ax.set_title(f"{name} of active codes: {method}", fontsize=14)
        ax.set_xlabel(f"{name} 1", fontsize=12)                 # axis names
        ax.set_ylabel(f"{name} 2", fontsize=12)                 # axis names
        ax.grid(color="#e5e5e5", linewidth=0.6)                 # light grid, as in 1_clustering
        ax.set_axisbelow(True)                                  # grid behind the points
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)                  # cleaner frame
    handles, texts = axes[r, 0].get_legend_handles_labels()     # this row's legend entries
    pairs = sorted(zip(texts, handles), key=lambda t: int(t[0].split()[1]))  # cluster-id order
    axes[r, 1].legend([h for _, h in pairs], [t for t, _ in pairs], loc="center left",
                      bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=11, markerscale=1.3)

fig.tight_layout()                                              # avoid overlapping titles
vis_dir = f"{log_dir}/morphology_kernel_kmeans_k{kk_clusters}"  # same folder as the kernel K-means tables
os.makedirs(vis_dir, exist_ok=True)                             # create it if the morphology cell hasn't run
fig.savefig(f"{vis_dir}/umap_tsne_kernel_kmeans_vs_kmeans.png", dpi=300, bbox_inches="tight")
plt.show()                                                      # render

In [ ]:
# --- 3D UMAP and 3D t-SNE of the active codes: kernel K-means vs K-means labels ----
import os                                                       # create the output folder if missing
import umap                                                     # umap-learn package
from sklearn.manifold import TSNE                               # t-SNE (supports up to 3 components)

X_vis = np.asarray(codes_active, dtype=np.float64)              # the exact matrix both methods clustered (419, 46)

# --- 1. project to 3D once (same settings as the 2D cell, only n_components changes) --
umap_3d_vis = umap.UMAP(n_neighbors=80, min_dist=0.0125, n_components=3, metric="euclidean",
                        random_state=42).fit_transform(X_vis)   # new fit, not the 2D map plus an axis
tsne_3d_vis = TSNE(n_components=3, perplexity=20, init="pca", random_state=0,
                   max_iter=1000).fit_transform(X_vis)          # new fit, not the 2D map plus an axis

# # --- 2. static 2 x 2 figure: rows = labelling, columns = projection --------------------
# fig = plt.figure(figsize=(18, 15))                              # room for four 3D panels
# for r, (ids, method) in enumerate(rows):                        # `rows` from the 2D cell: kernel K-means, K-means
#     ids = np.asarray(ids).astype(int)                           # plain integer labels
#     order = sorted(np.unique(ids), key=lambda c: -(ids == c).sum())  # big clusters first, small ones on top
#     for col, (coords, name) in enumerate([(umap_3d_vis, "UMAP"), (tsne_3d_vis, "t-SNE")]):
#         ax = fig.add_subplot(2, 2, 2 * r + col + 1, projection="3d")  # panel position 1..4
#         for c in order:                                         # draw cluster by cluster
#             m = ids == c                                        # members of cluster c
#             ax.scatter(coords[m, 0], coords[m, 1], coords[m, 2], c=palette[c], marker=markers[c],
#                        s=30 if m.sum() > 20 else 70,            # enlarge small clusters so they stay visible
#                        alpha=0.85, edgecolors="white", linewidths=0.4,
#                        label=f"cluster {c} (n={m.sum()})")
#         ax.set_title(f"3D {name} of active codes: {method}", fontsize=14)
#         ax.set_xlabel(f"{name} 1")                              # axis names
#         ax.set_ylabel(f"{name} 2")                              # axis names
#         ax.set_zlabel(f"{name} 3")                              # axis names
#         ax.view_init(elev=20, azim=40)                          # camera angle; drag to rotate with %matplotlib widget
#         ax.set_box_aspect(None, zoom=0.85)                      # shrink the box slightly so axis labels are not clipped
#         if col == 1:                                            # one legend per row, on the right panel
#             handles, texts = ax.get_legend_handles_labels()     # this row's legend entries
#             pairs = sorted(zip(texts, handles), key=lambda t: int(t[0].split()[1]))  # cluster-id order
#             ax.legend([h for _, h in pairs], [t for t, _ in pairs], loc="center left",
#                       bbox_to_anchor=(1.05, 0.5), frameon=False, fontsize=11)

# fig.tight_layout()                                              # avoid overlapping titles
# vis_dir = f"{log_dir}/morphology_kernel_kmeans_k{kk_clusters}"  # same folder as the 2D figure
# os.makedirs(vis_dir, exist_ok=True)                             # create it if missing
# fig.savefig(f"{vis_dir}/umap_tsne_3d_kernel_kmeans_vs_kmeans.png", dpi=300, bbox_inches="tight")
# plt.show()                                                      # render

# --- 3. interactive 3D: rotate with the mouse, hover a point to see the subject ------
import plotly.express as px                                     # interactive 3D scatter plots

for ids, method in rows:                                        # kernel K-means, then K-means
    ids = np.asarray(ids).astype(int)                           # plain integer labels
    ids_sorted = np.unique(ids)                                 # cluster ids present
    for coords, name in [(umap_3d_vis, "UMAP"), (tsne_3d_vis, "t-SNE")]:
        df = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1], "z": coords[:, 2],
                           "cluster": [f"cluster {c}" for c in ids],   # text, so plotly uses discrete colours
                           "case": subjects})                          # `subjects` from the morphology cell
        fig_i = px.scatter_3d(df, x="x", y="y", z="z", color="cluster", hover_name="case",
                              color_discrete_map={f"cluster {c}": palette[c] for c in ids_sorted},
                              category_orders={"cluster": [f"cluster {c}" for c in ids_sorted]},
                              labels={"x": f"{name} 1", "y": f"{name} 2", "z": f"{name} 3"},
                              title=f"3D {name} of active codes: {method} (hover for subject id)")
        fig_i.update_traces(marker=dict(size=4, opacity=0.85))  # smaller markers suit 3D
        fig_i.update_layout(width=850, height=700)              # similar size to the 2D plotly figures
        fig_i.show()                                            # render

In [ ]:
# --- colour-coded projections: 8 metrics x (kernel PCA, UMAP, t-SNE) x (2D, 3D) x 2 methods ---
# colour = metric value (same for both methods), marker shape = cluster (the only thing that differs)
import os                                                       # output folders
from matplotlib.lines import Line2D                             # legend entries for marker shapes

needed = ["metrics", "counts", "subjects", "kk_ids", "cluster_ids", "codes_kk_2d", "codes_kk_3d",
          "umap_2d", "tsne_2d", "umap_3d_vis", "tsne_3d_vis", "log_dir"]  # built by earlier cells
missing = [name for name in needed if name not in globals()]    # anything not run yet
assert not missing, f"run the earlier cells first, missing: {missing}"

# --- 1. one row-aligned table with all 8 metrics -------------------------------------
metric_table = metrics.merge(counts, on="subject", how="left", validate="one_to_one")  # 6 means + 2 counts
assert metric_table["subject"].tolist() == subjects, "metric rows not in latent-code order"
metric_columns = [c for c in metric_table.columns if c != "subject"]  # the 8 metric names
print("metrics:", metric_columns)                               # expect 8 names

# --- 2. what to draw, in the order it is drawn ----------------------------------------
methods = [("kernel_kmeans", kk_ids, f"kernel K-means (K={kk_clusters})"),   # grouped by method first
           ("kmeans", cluster_ids, f"K-means (K={n_clusters})")]
projections = [("kernel_pca", "Kernel PCA", "Kernel PC", {"2D": codes_kk_2d, "3D": codes_kk_3d}),
               ("umap", "UMAP", "UMAP ", {"2D": umap_2d, "3D": umap_3d_vis}),
               ("tsne", "t-SNE", "t-SNE ", {"2D": tsne_2d, "3D": tsne_3d_vis})]
cluster_markers = ["o"] * 8      # one shape per cluster id
out_root = f"{log_dir}/metric_projections"                      # one sub-folder per clustering method
save_single_panels = False                                      # True also saves 96 one-metric PNGs (slow)


def draw_metric_panel(fig, grid, position, coords, values, ids, metric, axis_prefix, is_3d):
    """One panel: points placed by the projection, coloured by a metric, shaped by cluster."""
    ax = fig.add_subplot(*grid, position, projection="3d" if is_3d else None)  # 2D or 3D axes
    vmin, vmax = np.nanpercentile(values, [2, 98])             # clip extremes so they don't flatten the scale
    n_dims = 3 if is_3d else 2                                  # how many coordinates to pass
    missing_value = np.isnan(values)                            # shapes without this metric
    if missing_value.any():                                     # draw them grey, otherwise they vanish
        ax.scatter(*coords[missing_value, :n_dims].T, c="lightgrey", s=12, marker="o")
    for c in np.unique(ids):                                    # one call per cluster = one marker shape
        m = (ids == c) & ~missing_value                         # cluster members with a value
        points = ax.scatter(*coords[m, :n_dims].T, c=values[m], cmap="viridis", vmin=vmin, vmax=vmax,
                            marker=cluster_markers[c], s=18, alpha=0.9,
                            edgecolors="#333333", linewidths=0.2)  # thin edge keeps shapes readable
      
    fig.colorbar(points, ax=ax, shrink=0.55 if is_3d else 0.7, pad=0.09 if is_3d else 0.02)  # metric scale
    ax.set_title(metric, fontsize=12)                           # which metric colours this panel
    ax.set_xlabel(f"{axis_prefix}1", fontsize=9)                # axis names
    ax.set_ylabel(f"{axis_prefix}2", fontsize=9)                # axis names
    ax.tick_params(labelsize=7)                                 # small ticks, many panels
    if is_3d:
        ax.set_zlabel(f"{axis_prefix}3", fontsize=9)            # third axis name
        ax.view_init(elev=20, azim=40)                          # same camera angle as the other 3D plots
        ax.set_box_aspect(None, zoom=0.95)                      # keep axis labels inside the panel
    return ax


# --- 3. method -> projection -> 2D/3D, one 2 x 4 figure (8 metrics) each --------------
for method_key, ids, method_label in methods:                   # 1st level: clustering method
    ids = np.asarray(ids).astype(int)                           # plain integer labels
    method_dir = f"{out_root}/{method_key}"                     # e.g. .../metric_projections/kmeans
    os.makedirs(method_dir, exist_ok=True)                      # create it if missing
    shape_legend = [Line2D([], [], linestyle="", marker=cluster_markers[c], color="grey",
                           markersize=9, label=f"cluster {c} (n={(ids == c).sum()})")
                    for c in np.unique(ids)]                    # marker key, shared by this method's figures

    for proj_key, proj_label, axis_prefix, coords_by_dim in projections:  # 2nd level: projection
        for dim in ["2D", "3D"]:                                # 3rd level: 2D then 3D
            coords = np.asarray(coords_by_dim[dim])             # (419, 2) or (419, 3)
            is_3d = dim == "3D"                                 # choose the axes type
            fig = plt.figure(figsize=(24, 12) if is_3d else (22, 10))  # 3D panels need more room
            for i, metric in enumerate(metric_columns):         # one panel per metric
                values = metric_table[metric].to_numpy(dtype=float)  # metric in latent row order
                draw_metric_panel(fig, (2, 4), i + 1, coords, values, ids, metric, axis_prefix, is_3d)
            fig.suptitle(f"{method_label}  |  {proj_label} {dim}  |  colour = metric", fontsize=16)
            fig.legend(handles=shape_legend, loc="lower center", ncol=len(shape_legend),
                       frameon=False, fontsize=12)              # which shape is which cluster
            fig.tight_layout(rect=[0, 0.05, 1, 0.95])           # leave room for title and legend
            path = f"{method_dir}/{proj_key}_{dim}_8metrics.png"  # e.g. kmeans/umap_2D_8metrics.png
            fig.savefig(path, dpi=200, bbox_inches="tight")     # one overview file per combination
            print(path)                                         # where it went
            plt.show()                                          # render

            if save_single_panels:                              # optional: one PNG per metric
                panel_dir = f"{method_dir}/{proj_key}_{dim}"    # keep the 8 files together
                os.makedirs(panel_dir, exist_ok=True)           # create it if missing
                for metric in metric_columns:                   # one file per metric
                    single = plt.figure(figsize=(8, 7) if is_3d else (7, 6))  # single-panel canvas
                    values = metric_table[metric].to_numpy(dtype=float)
                    draw_metric_panel(single, (1, 1), 1, coords, values, ids, metric, axis_prefix, is_3d)
                    single.legend(handles=shape_legend, loc="lower center", ncol=2, frameon=False)
                    single.tight_layout(rect=[0, 0.08, 1, 1])   # room for the legend
                    single.savefig(f"{panel_dir}/{metric}.png", dpi=200, bbox_inches="tight")
                    plt.close(single)                           # don't display 96 extra figures


# # Parameter sweep: UMAP n_neighbors and t-SNE perplexity = 5, 10, ..., 55 (2D and 3D)

In [ ]:
# --- sweep: 2 methods x (UMAP, t-SNE) x 11 settings x (2D, 3D), coloured by cluster and by 8 metrics ---
# Run after the colour-coded projections cell (it defines metric_table, draw_metric_panel, ...).
import os                                                       # output folders
import umap                                                     # umap-learn package
from sklearn.manifold import TSNE                               # t-SNE (up to 3 components)
from matplotlib.figure import Figure                            # outside pyplot: no widget, no leak
from matplotlib.lines import Line2D                             # legend entries for marker shapes
from IPython.display import Image, display                      # preview saved PNGs inline

needed = ["codes_active", "kk_ids", "cluster_ids", "kk_clusters", "n_clusters", "metric_table",
          "metric_columns", "draw_metric_panel", "cluster_markers", "palette", "log_dir"]  # earlier cells
missing = [name for name in needed if name not in globals()]    # anything not run yet
assert not missing, f"run the earlier cells first, missing: {missing}"  # stop before fitting anything

sweep_values = list(range(5, 60, 5))                            # 5, 10, ..., 55 (stops before 60)
dims = {"2D": 2, "3D": 3}                                       # label -> number of components
X_vis = np.asarray(codes_active, dtype=np.float64)              # same matrix as the other projections
out_root = f"{log_dir}/metric_projections"                      # same root as the default figures
cache_path = f"{out_root}/sweep_embeddings.npz"                 # delete this file to force a refit
sweep_dpi = 200                                                 # as the default figures; lower = faster
show_overviews = True                                           # display the 8 overviews at the end
os.makedirs(out_root, exist_ok=True)                            # create it if missing

methods = [("kernel_kmeans", kk_ids, f"kernel K-means (K={kk_clusters})"),  # folder, labels, title
           ("kmeans", cluster_ids, f"K-means (K={n_clusters})")]


def fit_umap(n_neighbors, n_components):
    """UMAP with the 2D cell's settings; only n_neighbors and n_components change."""
    model = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.0125, n_components=n_components,
                      metric="euclidean", random_state=42)      # fixed seed, reproducible
    return model.fit_transform(X_vis)                           # (419, n_components)


def fit_tsne(perplexity, n_components):
    """t-SNE with the 2D cell's settings; only perplexity and n_components change."""
    model = TSNE(n_components=n_components, perplexity=perplexity, init="pca",
                 random_state=0, max_iter=1000)                 # fixed seed, reproducible
    return model.fit_transform(X_vis)                           # (419, n_components)


# key, title, axis prefix, parameter name, file tag, fitting function
sweeps = [("umap", "UMAP", "UMAP ", "n_neighbors", "nn", fit_umap),
          ("tsne", "t-SNE", "t-SNE ", "perplexity", "perp", fit_tsne)]


def embedding_name(proj_key, tag, value, dim):
    """Cache key and file stem, e.g. umap_nn05_2D."""
    return f"{proj_key}_{tag}{value:02d}_{dim}"                 # files sort 05, 10 (:02d = zero-pad)


# --- 1. fit the 44 embeddings once, reuse them on later runs ---------------------------
embeddings = {}                                                 # name -> (419, 2) or (419, 3) array
if os.path.exists(cache_path):                                  # an earlier run saved its fits
    with np.load(cache_path) as cached:                         # npz archive, closed after the block
        embeddings = dict(cached)                               # copy every array into a plain dict
    stale = [k for k, v in embeddings.items() if len(v) != len(X_vis)]  # fits of other data
    assert not stale, f"cache does not match codes_active, delete {cache_path}"  # refuse to mix

for proj_key, proj_label, _, param_name, tag, fit in sweeps:    # UMAP, then t-SNE
    for value in sweep_values:                                  # 5 ... 55
        for dim, n_components in dims.items():                  # 2D, then 3D
            name = embedding_name(proj_key, tag, value, dim)    # e.g. tsne_perp30_3D
            if name in embeddings:                              # fitted on an earlier run
                continue                                        # skip the refit
            print(f"fitting {proj_label} {dim}, {param_name}={value}")  # progress
            embeddings[name] = fit(value, n_components)         # own fit, not the 2D map plus an axis
np.savez(cache_path, **embeddings)                              # save all (** = dict as keyword args)
print(f"{len(embeddings)} embeddings in {cache_path}")         # expect 44


# --- 2. helpers for the cluster-coloured overview -----------------------------------------
def cluster_legend(ids):
    """Colour + marker key for the clusters of one method."""
    handles = []                                                # one entry per cluster
    for c in np.unique(ids):                                    # cluster-id order
        handles.append(Line2D([], [], linestyle="", marker=cluster_markers[c], color=palette[c],
                              markersize=9, label=f"cluster {c} (n={(ids == c).sum()})"))  # key
    return handles                                              # passed to fig.legend


def draw_cluster_overview(ids, method_label, proj_key, proj_label, axis_prefix, param_name, tag, dim):
    """One figure, 11 panels: the same projection at every setting, coloured by cluster."""
    is_3d = dim == "3D"                                         # choose the axes type
    n_dims = dims[dim]                                          # coordinates to pass to scatter
    fig = Figure(figsize=(24, 18) if is_3d else (22, 15))       # 3 x 4 grid, 11 panels used
    order = sorted(np.unique(ids), key=lambda c: -(ids == c).sum())  # big first, small on top
    for i, value in enumerate(sweep_values):                    # one panel per setting
        coords = embeddings[embedding_name(proj_key, tag, value, dim)]  # this setting's map
        ax = fig.add_subplot(3, 4, i + 1, projection="3d" if is_3d else None)  # 2D or 3D axes
        for c in order:                                         # draw cluster by cluster
            m = ids == c                                        # members of cluster c
            # *coords[...].T passes the columns as separate x, y(, z) arguments
            ax.scatter(*coords[m, :n_dims].T, c=palette[c], marker=cluster_markers[c],
                       s=14, alpha=0.85, edgecolors="white", linewidths=0.3)
        ax.set_title(f"{param_name} = {value}", fontsize=12)    # which setting this panel shows
        ax.set_xlabel(f"{axis_prefix}1", fontsize=9)            # axis names
        ax.set_ylabel(f"{axis_prefix}2", fontsize=9)            # axis names
        ax.tick_params(labelsize=7)                             # small ticks, many panels
        if is_3d:                                               # extra settings for 3D panels
            ax.set_zlabel(f"{axis_prefix}3", fontsize=9)        # third axis name
            ax.view_init(elev=20, azim=40)                      # same camera as the other 3D plots
    title = f"{method_label}  |  {proj_label} {dim}  |  {param_name} sweep, colour = metric"
    fig.suptitle(title, fontsize=16)                            # method, projection, dimension
    fig.legend(handles=cluster_legend(ids), loc="lower right", bbox_to_anchor=(0.97, 0.12),
               frameon=False, fontsize=13)                      # sits in the empty 12th slot
    fig.tight_layout(rect=[0, 0, 1, 0.96])                      # leave room for the title
    return fig                                                  # caller saves it


# --- 3. method -> projection -> 2D/3D -> setting: overview + one 8-metric figure each ---
overview_paths = []                                             # previewed at the end
for method_key, ids, method_label in methods:                   # 1st level: clustering method
    ids = np.asarray(ids).astype(int)                           # plain integer labels
    shape_legend = [Line2D([], [], linestyle="", marker=cluster_markers[c], color="grey",
                           markersize=9, label=f"cluster {c} (n={(ids == c).sum()})")
                    for c in np.unique(ids)]                    # marker key, as in the default figures

    for proj_key, proj_label, axis_prefix, param_name, tag, _ in sweeps:  # 2nd level: projection
        sweep_dir = f"{out_root}/{method_key}/{proj_key}_sweep"  # e.g. .../kmeans/umap_sweep
        os.makedirs(sweep_dir, exist_ok=True)                   # create it if missing

        for dim in dims:                                        # 3rd level: 2D then 3D
            is_3d = dim == "3D"                                 # choose the axes type

            overview = draw_cluster_overview(ids, method_label, proj_key, proj_label,
                                             axis_prefix, param_name, tag, dim)  # 11 settings
            path = f"{sweep_dir}/{proj_key}_{dim}_clusters_all_{param_name}.png"  # overview file
            overview.savefig(path, dpi=sweep_dpi, bbox_inches="tight")  # write it
            overview_paths.append(path)                         # remember it for the preview

            for value in sweep_values:                          # 4th level: parameter setting
                coords = embeddings[embedding_name(proj_key, tag, value, dim)]  # this setting's map
                fig = Figure(figsize=(24, 12) if is_3d else (22, 10))  # as the default figures
                for i, metric in enumerate(metric_columns):     # one panel per metric
                    values = metric_table[metric].to_numpy(dtype=float)  # latent row order
                    draw_metric_panel(fig, (2, 4), i + 1, coords, values, ids, metric,
                                      axis_prefix, is_3d)       # same panel style as the defaults
                title = (f"{method_label}  |  {proj_label} {dim}, {param_name}={value}"
                         f"  |  colour = metric, marker = cluster")  # two f-strings, joined
                fig.suptitle(title, fontsize=16)                # method, projection, setting
                fig.legend(handles=shape_legend, loc="lower center", ncol=len(shape_legend),
                           frameon=False, fontsize=12)          # which shape is which cluster
                fig.tight_layout(rect=[0, 0.05, 1, 0.95])       # leave room for title and legend
                stem = embedding_name(proj_key, tag, value, dim)  # e.g. umap_nn05_2D
                path = f"{sweep_dir}/{stem}_8metrics.png"       # one file per setting
                fig.savefig(path, dpi=sweep_dpi, bbox_inches="tight")  # write it, never displayed

            print(f"saved {dim} {proj_label} sweep for {method_label} in {sweep_dir}")  # progress

# for proj_key, proj_label, axis_prefix, param_name, tag, _ in sweeps:  # UMAP, then t-SNE
#     for value in sweep_values:                                  # 5 ... 55
#         coords_2d = embeddings[embedding_name(proj_key, tag, value, "2D")]  # this setting in 2D
#         coords_3d = embeddings[embedding_name(proj_key, tag, value, "3D")]  # and in 3D
#         ex.add("K-means", proj_label, cluster_ids, coords_2d=coords_2d, coords_3d=coords_3d,
#                axes=axis_prefix, parameter=param_name, setting=value)  # K-means tab
#         ex.add("Kernel K-means", proj_label, kk_ids, coords_2d=coords_2d, coords_3d=coords_3d,
#                axes=axis_prefix, parameter=param_name, setting=value)  # kernel K-means tab

# ex.save(f"{log_dir}/latent_explorer.html")                     # overwrites the default-only file
# --- 4. look at the cluster overviews first; open the 8-metric PNGs for settings you like ---
if show_overviews:                                              # switch off for silent re-runs
    for path in overview_paths:                                 # 2 methods x 2 projections x 2 dims
        print(path)                                             # which one follows
        display(Image(filename=path, width=1100))               # scaled down to fit the notebook

In [ ]:
# --- box plots per cluster, all 8 metrics: K-means and kernel K-means ----------------
import os                                                             # create output folders if missing
import numpy as np                                                    # arrays
import pandas as pd                                                   # tables keyed by subject
import matplotlib.pyplot as plt                                       # plotting


def plot_cluster_boxplots(cluster_labels, metrics, counts, method_name, out_dir):
    """One box per cluster for each of the 8 metrics; tick labels show n, mean and variance."""
    table = metrics.merge(counts, on="subject", how="left")          # 6 means + 2 counts (left = keep row order)
    table["cluster"] = np.asarray(cluster_labels).astype(int)        # cluster id per subject, same row order
    metric_columns = [c for c in table.columns if c not in ("subject", "cluster")]  # the 8 metrics
    clusters = np.unique(table["cluster"])                           # sorted cluster ids

    n_cols = 4                                                       # 4 panels per row
    n_rows = int(np.ceil(len(metric_columns) / n_cols))              # 8 metrics -> 2 rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
    axes = np.atleast_1d(axes).ravel()                               # flat list of panels (ravel = 2D -> 1D)

    stat_rows = []                                                   # mean / variance per cluster, for printing
    for ax, metric in zip(axes, metric_columns):                     # one panel per metric
        groups = []                                                  # values of each cluster
        tick_text = []                                               # label under each box
        for c in clusters:                                           # one box per cluster
            values = table.loc[table["cluster"] == c, metric].dropna().to_numpy(dtype=float)
            groups.append(values)                                    # data for this box
            mean = values.mean() if len(values) else np.nan          # cluster mean
            var = values.var(ddof=1) if len(values) > 1 else np.nan  # sample variance (ddof=1 = divide by n-1)
            tick_text.append(f"cluster {c}\nn={len(values)}\nμ={mean:.3g}\nσ²={var:.3g}")
            stat_rows.append({"metric": metric, "cluster": c, "n": len(values),
                              "mean": mean, "variance": var, "sd": np.sqrt(var)})

        positions = np.arange(1, len(clusters) + 1)                  # x position of each box
        ax.boxplot(groups, positions=positions, widths=0.6,
                   showmeans=True, meanline=False,                   # draw the mean as a marker
                   meanprops=dict(marker="D", markerfacecolor="red", markeredgecolor="black", markersize=7),
                   medianprops=dict(color="black", linewidth=1.5),   # median = black line
                   flierprops=dict(marker="o", markersize=3, alpha=0.4))  # outliers, faint

        for pos, values in zip(positions, groups):                   # overlay every subject as a dot
            jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(values))  # spread dots sideways
            ax.scatter(pos + jitter, values, s=6, alpha=0.3, color="grey", zorder=0)  # zorder=0 = behind boxes

        ax.set_xticks(positions)                                     # one tick per box
        ax.set_xticklabels(tick_text, fontsize=9)                    # n, mean and variance under each box
        ax.set_title(metric, fontsize=13)                            # which metric this panel shows
        ax.grid(axis="y", color="#e5e5e5", linewidth=0.6)            # light horizontal grid
        ax.set_axisbelow(True)                                       # grid behind the boxes

    for ax in axes[len(metric_columns):]:                            # hide unused panels, if any
        ax.set_visible(False)

    fig.suptitle(f"{method_name}: metrics per cluster (red diamond = mean, line = median)", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])                           # leave room for the title
    path = f"{out_dir}/boxplots_{method_name.replace(' ', '_').lower()}.png"  # e.g. boxplots_k-means.png
    fig.savefig(path, dpi=300, bbox_inches="tight")                  # write the PNG
    plt.show()                                                       # render

    stats = pd.DataFrame(stat_rows)                                  # long table: metric x cluster
    stats_wide = stats.pivot(index="cluster", columns="metric", values=["mean", "variance"])  # one row per cluster
    print(path)                                                      # where the figure went
    return stats_wide                                                # so you can compare with your table


# --- run it for both methods, each into its own existing folder ------------------------
os.makedirs(kmeans_dir, exist_ok=True)                               # .../morphology_kmeans_k4
os.makedirs(kernel_dir, exist_ok=True)                               # .../morphology_kernel_kmeans_k4

kmeans_box_stats = plot_cluster_boxplots(cluster_ids, metrics, counts, "K-means", kmeans_dir)
kk_box_stats = plot_cluster_boxplots(kk_ids, metrics, counts, "Kernel K-means", kernel_dir)

display(kmeans_box_stats.round(3))                                   # K-means mean + variance per cluster
display(kk_box_stats.round(3))                                       # kernel K-means mean + variance per cluster

In [ ]:
import numpy as np                                                   # reads the sweep cache
import build_explorer as bx                                          # the explorer builder
 
ex = bx.Explorer("Airway latent space explorer", subjects)           # same subjects, same order
ex.load(f"{log_dir}/morphology/explorer_ward.json")                  # agglomerative tab, and the metrics
 
TABS = [("K-means", cluster_ids), ("Kernel K-means", kk_ids)]        # both read the same coordinates
 
for tab, ids in TABS:                                                # kernel PCA has one gamma, no menu
    ex.add(tab, "Kernel PCA", ids, coords_2d=codes_kk_2d, coords_3d=codes_kk_3d, axes="Kernel PC")
 
if "embeddings" not in globals():                                    # sweep cell not run this session
    with np.load(f"{log_dir}/metric_projections/sweep_embeddings.npz") as cached:
        embeddings = dict(cached)                                    # the 44 fits it saved
print(f"{len(embeddings)} sweep embeddings available")               # expect 44
 
SWEEPS = [("umap", "UMAP", "UMAP ", "n_neighbors", "nn"),            # key, tab, axes, parameter, file tag
          ("tsne", "t-SNE", "t-SNE ", "perplexity", "perp")]
 
 
def sweep_settings(proj_key, tag):
    """The parameter values present in `embeddings`, e.g. [5, 10, ..., 55]."""
    values = set()                                                   # collected from the cache keys
    for key in embeddings:                                           # keys look like umap_nn05_2D
        parts = key.split("_")                                       # ['umap', 'nn05', '2D']
        if parts[0] == proj_key and parts[1].startswith(tag):
            values.add(int(parts[1][len(tag):]))                     # strip the tag, keep the number
    return sorted(values)
 
 
for proj_key, proj_label, axes, parameter, tag in SWEEPS:            # UMAP, then t-SNE
    values = sweep_settings(proj_key, tag)                           # every setting that was fitted
    assert values, f"no {proj_label} embeddings in the sweep cache"
    print(f"{proj_label}: {parameter} = {values}")                   # what goes into the menu
    for value in values:                                             # one menu entry per setting
        coords_2d = embeddings.get(f"{proj_key}_{tag}{value:02d}_2D")   # this setting in 2D
        coords_3d = embeddings.get(f"{proj_key}_{tag}{value:02d}_3D")   # and in 3D
        for tab, ids in TABS:                                        # same map, the two labellings
            ex.add(tab, proj_label, ids, coords_2d=coords_2d, coords_3d=coords_3d,
                   axes=axes, parameter=parameter, setting=value)
 
ex.save(f"{log_dir}/latent_explorer.html")                           # one file, opens offline
 
